# Distanzmetriken & Normalisierung

Distanzmetriken sind das Herzstück vieler ML-Algorithmen: k-Nearest Neighbors, Clustering (k-Means, DBSCAN), Empfehlungssysteme und semantische Suche. Die Wahl der richtigen Metrik entscheidet über die Qualität des Modells.

In diesem Notebook:
1. **Euklidische Distanz** – der geometrische Abstand
2. **Cosinus-Ähnlichkeit & -Distanz** – der Winkel zwischen Vektoren
3. **Min-Max-Normalisierung** – Werte auf [0, 1] skalieren
4. **Z-Score-Standardisierung** – Mittelwert 0, Standardabweichung 1
5. **Praxis: Warum Normalisierung vor Distanzberechnung wichtig ist**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Die Algorithmen

In [ ]:
def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Euklidische Distanz: ||a - b||₂"""
    return np.sqrt(np.sum((a - b) ** 2))


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosinus-Ähnlichkeit: a·b / (||a||·||b||)"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)


def cosine_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Cosinus-Distanz: 1 - cosine_similarity"""
    return 1 - cosine_similarity(a, b)


def normalize(x: np.ndarray) -> np.ndarray:
    """Min-Max-Normalisierung auf [0, 1]."""
    x_min, x_max = x.min(), x.max()
    if abs(x_max - x_min) < 1e-10:
        return np.zeros_like(x)
    return (x - x_min) / (x_max - x_min)


def standardize(x: np.ndarray) -> np.ndarray:
    """Z-Score-Standardisierung: (x - μ) / σ"""
    return (x - x.mean()) / (x.std() + 1e-8)

## 2. Euklidische Distanz

Die euklidische Distanz ist der „normale" Abstand – die Länge der geraden Linie zwischen zwei Punkten:

$$d(a, b) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

Das ist der Satz des Pythagoras in $n$ Dimensionen.

In [ ]:
# Euklidische Distanz: Beispiele
a = np.array([0.0, 0.0])
b = np.array([3.0, 4.0])   # 3-4-5-Dreieck!
c = np.array([1.0, 1.0])

print(f"d(a, b) = {euclidean_distance(a, b):.4f}  (3-4-5-Dreieck: √(3²+4²) = 5)")
print(f"d(a, c) = {euclidean_distance(a, c):.4f}  (√(1²+1²) = √2)")
print(f"d(b, c) = {euclidean_distance(b, c):.4f}")
print()

# Höherdimensional
x = np.array([1.0, 0.0, 0.0, 0.0])
y = np.array([0.0, 1.0, 0.0, 0.0])
z = np.array([0.0, 0.0, 1.0, 0.0])
print(f"4D: d(x, y) = {euclidean_distance(x, y):.4f}  (√2)")
print(f"4D: d(x, z) = {euclidean_distance(x, z):.4f}  (√2)")
print(f"4D: d(y, z) = {euclidean_distance(y, z):.4f}  (√2)")

In [ ]:
# Visualisierung: Euklidische Distanz in 2D
fig, ax = plt.subplots(figsize=(8, 7))

points = {
    'A': np.array([0.0, 0.0]),
    'B': np.array([3.0, 4.0]),
    'C': np.array([1.0, 1.0]),
    'D': np.array([5.0, 1.0]),
    'E': np.array([2.0, 3.0])
}

colors = {'A': 'red', 'B': 'blue', 'C': 'green', 'D': 'orange', 'E': 'purple'}

for name, pt in points.items():
    ax.scatter(*pt, c=colors[name], s=100, zorder=5)
    ax.annotate(f'{name} ({pt[0]}, {pt[1]})', pt, textcoords="offset points",
                xytext=(10, 10), fontsize=11, fontweight='bold')

# Verbindungen mit Distanzen
pairs = [('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'D'), ('D', 'E')]
for p1, p2 in pairs:
    d = euclidean_distance(points[p1], points[p2])
    mid = (points[p1] + points[p2]) / 2
    ax.plot([points[p1][0], points[p2][0]], [points[p1][1], points[p2][1]],
            'gray', linestyle='--', alpha=0.6, linewidth=1.5)
    ax.annotate(f'{d:.2f}', mid, textcoords="offset points",
                xytext=(0, -15), fontsize=9, ha='center', color='gray')

ax.set_xlim(-1, 7)
ax.set_ylim(-1, 6)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Euklidische Distanzen zwischen Punkten')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 3. Cosinus-Ähnlichkeit & -Distanz

Die Cosinus-Ähnlichkeit misst den **Winkel** zwischen zwei Vektoren, nicht deren Länge:

$$\text{sim}(a, b) = \frac{a \cdot b}{\|a\| \cdot \|b\|} = \cos(\theta)$$

- **+1**: gleiche Richtung (identisch bis auf Skalierung)
- **0**: orthogonal (keine Ähnlichkeit)
- **-1**: entgegengesetzte Richtung

Die Cosinus-Distanz ist $1 - \text{sim}(a, b)$, also im Bereich $[0, 2]$.

In [ ]:
# Cosinus-Ähnlichkeit: Beispiele
a = np.array([1.0, 0.0, 0.0])   # x-Achse
b = np.array([0.0, 1.0, 0.0])   # y-Achse (orthogonal)
c = np.array([1.0, 0.0, 0.0])   # identisch zu a
d = np.array([2.0, 0.0, 0.0])   # gleiche Richtung, andere Länge
e = np.array([-1.0, 0.0, 0.0])  # entgegengesetzt

print("Cosinus-Ähnlichkeit:")
print(f"  sim(a, b) = {cosine_similarity(a, b):.4f}  (orthogonal → 0)")
print(f"  sim(a, c) = {cosine_similarity(a, c):.4f}  (identisch → 1)")
print(f"  sim(a, d) = {cosine_similarity(a, d):.4f}  (gleiche Richtung, skaliert → 1)")
print(f"  sim(a, e) = {cosine_similarity(a, e):.4f}  (entgegengesetzt → -1)")
print()
print("Cosinus-Distanz:")
print(f"  dist(a, b) = {cosine_distance(a, b):.4f}  (1 - 0 = 1)")
print(f"  dist(a, c) = {cosine_distance(a, c):.4f}  (1 - 1 = 0)")
print(f"  dist(a, e) = {cosine_distance(a, e):.4f}  (1 - (-1) = 2)")

In [ ]:
# Visualisierung: Cosinus-Ähnlichkeit in 2D
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

scenarios = [
    (np.array([1.0, 0.0]), np.array([0.0, 1.0]), 'Orthogonal (90°)'),
    (np.array([1.0, 0.0]), np.array([0.707, 0.707]), '45° Winkel'),
    (np.array([1.0, 0.0]), np.array([0.966, 0.259]), '15° Winkel'),
]

for ax, (v1, v2, title) in zip(axes, scenarios):
    # Vektoren vom Ursprung
    ax.quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1,
              color='blue', width=0.02, label='Vektor a')
    ax.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1,
              color='red', width=0.02, label='Vektor b')

    sim = cosine_similarity(v1, v2)
    angle = np.arccos(np.clip(sim, -1, 1))

    # Winkelbogen
    theta_vals = np.linspace(0, angle, 50)
    arc_r = 0.3
    ax.plot(arc_r * np.cos(theta_vals), arc_r * np.sin(theta_vals),
            'green', linewidth=2)
    ax.text(0.4, 0.15, f'{np.degrees(angle):.0f}°', fontsize=11, color='green')

    ax.set_xlim(-0.2, 1.3)
    ax.set_ylim(-0.2, 1.3)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'{title}\nsim = {sim:.3f}')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('Cosinus-Ähnlichkeit = cos(Winkel)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Euklidisch vs. Cosinus – Wann welche?

| Metrik | Misst | Gut für | Schlecht für |
|--------|-------|---------|--------------|
| **Euklidisch** | Absoluten Abstand | Koordinaten, Sensordaten, Clustering | Text (Sparsity, Längeneffekt) |
| **Cosinus** | Richtung / Winkel | Text (TF-IDF), Embeddings, Empfehlungen | Wenn Betrag wichtig ist |

**Faustregel:** Cosinus für hochdimensionale, sparse Daten (Text). Euklidisch für dichte, niedrigdimensionale Daten (Sensoren, Koordinaten).

## 4. Normalisierung & Standardisierung

Features mit unterschiedlichen Skalen verzerren Distanzberechnungen. Ein Feature in Metern und eins in Millimetern – die euklidische Distanz wird vom Millimeter-Feature dominiert.

### Min-Max-Normalisierung
$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
Skaliert alle Werte auf $[0, 1]$.

### Z-Score-Standardisierung
$$x_{\text{std}} = \frac{x - \mu}{\sigma}$$
Zentriert auf Mittelwert 0 mit Standardabweichung 1.

In [ ]:
# Demo: Warum Normalisierung wichtig ist
np.random.seed(42)

# Zwei Features mit extrem unterschiedlichen Skalen
groesse_cm = np.random.normal(170, 15, 100)     # Körpergröße in cm (Mittel ~170)
gewicht_kg = np.random.normal(70, 12, 100)      # Gewicht in kg (Mittel ~70)
einkommen  = np.random.normal(50000, 15000, 100) # Einkommen in € (Mittel ~50000!)

data_raw = np.column_stack([groesse_cm, gewicht_kg, einkommen])

print("Rohdaten – Statistiken:")
print(f"  Größe:    μ={groesse_cm.mean():.1f}, σ={groesse_cm.std():.1f}, Bereich=[{groesse_cm.min():.0f}, {groesse_cm.max():.0f}]")
print(f"  Gewicht:  μ={gewicht_kg.mean():.1f}, σ={gewicht_kg.std():.1f}, Bereich=[{gewicht_kg.min():.0f}, {gewicht_kg.max():.0f}]")
print(f"  Einkommen: μ={einkommen.mean():.1f}, σ={einkommen.std():.1f}, Bereich=[{einkommen.min():.0f}, {einkommen.max():.0f}]")

In [ ]:
# Normalisierung und Standardisierung anwenden
data_norm = np.apply_along_axis(normalize, 0, data_raw)
data_std  = np.apply_along_axis(standardize, 0, data_raw)

# Distanz zwischen Person 0 und Person 1 – vor und nach Skalierung
p0, p1 = 0, 1

d_raw  = euclidean_distance(data_raw[p0], data_raw[p1])
d_norm = euclidean_distance(data_norm[p0], data_norm[p1])
d_std  = euclidean_distance(data_std[p0], data_std[p1])

print("Euklidische Distanz zwischen Person 0 und 1:")
print(f"  Rohdaten:            {d_raw:.2f}  ← vom Einkommen dominiert!")
print(f"  Min-Max-normalisiert: {d_norm:.4f}")
print(f"  Z-Score-standardisiert: {d_std:.4f}")
print()
print("→ Ohne Normalisierung dominiert das Feature mit der größten Skala die Distanz!")

In [ ]:
# Visualisierung: Rohdaten vs. standardisiert
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Rohdaten (Größe vs. Einkommen)
axes[0].scatter(groesse_cm, einkommen, alpha=0.6, c='steelblue', edgecolors='white')
axes[0].set_xlabel('Größe (cm)')
axes[0].set_ylabel('Einkommen (€)')
axes[0].set_title('Rohdaten: Größe vs. Einkommen\n(Einkommen dominiert die Skala)')
axes[0].grid(True, alpha=0.3)

# Standardisiert
axes[1].scatter(standardize(groesse_cm), standardize(einkommen), alpha=0.6, c='darkorange', edgecolors='white')
axes[1].set_xlabel('Größe (z-score)')
axes[1].set_ylabel('Einkommen (z-score)')
axes[1].set_title('Standardisiert: Größe vs. Einkommen\n(Beide Features gleich gewichtet)')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.3)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.3)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Praxis: Distanzmatrix

Eine Distanzmatrix zeigt paarweise Distanzen zwischen allen Punkten – die Grundlage für Clustering und Nearest-Neighbor-Suche.

In [ ]:
# Distanzmatrix für 5 zufällige 2D-Punkte
np.random.seed(123)
points = np.random.randn(5, 2) * 2
n = len(points)

# Euklidische Distanzmatrix
dist_euclidean = np.zeros((n, n))
dist_cosine = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        dist_euclidean[i, j] = euclidean_distance(points[i], points[j])
        dist_cosine[i, j] = cosine_distance(points[i], points[j])

print("Euklidische Distanzmatrix:")
print(np.array2string(dist_euclidean, precision=2, suppress_small=True))
print()
print("Cosinus-Distanzmatrix:")
print(np.array2string(dist_cosine, precision=2, suppress_small=True))

In [ ]:
# Visualisierung: Punkte + Distanzen
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

labels = [f'P{i}' for i in range(n)]
colors = plt.cm.tab10(np.linspace(0, 1, n))

for ax, dist_matrix, title in zip(
    axes,
    [dist_euclidean, dist_cosine],
    ['Euklidische Distanz', 'Cosinus-Distanz']
):
    # Punkte plotten
    for i in range(n):
        ax.scatter(points[i, 0], points[i, 1], c=[colors[i]], s=150, zorder=5, edgecolors='white')
        ax.annotate(labels[i], points[i], textcoords="offset points",
                    xytext=(8, 8), fontsize=11, fontweight='bold')

    # Nächsten Nachbarn für jeden Punkt einzeichnen
    for i in range(n):
        dists = dist_matrix[i].copy()
        dists[i] = np.inf  # sich selbst ignorieren
        nearest = np.argmin(dists)
        ax.plot([points[i, 0], points[nearest, 0]],
                [points[i, 1], points[nearest, 1]],
                'gray', linestyle=':', alpha=0.5, linewidth=1)

    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'{title}\n(gestrichelt = nächster Nachbar)')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 6. Zusammenfassung

| Konzept | Formel | Wann verwenden? |
|---------|--------|-----------------|
| **Euklidische Distanz** | $\sqrt{\sum (a_i - b_i)^2}$ | Dichte, niedrigdimensionale Daten |
| **Cosinus-Ähnlichkeit** | $\frac{a \cdot b}{\|a\| \|b\|}$ | Text, Embeddings, sparse Vektoren |
| **Min-Max-Normalisierung** | $\frac{x - \min}{\max - \min}$ | Neuronale Netze, wenn [0,1] gewünscht |
| **Z-Score-Standardisierung** | $\frac{x - \mu}{\sigma}$ | Clustering, PCA, wenn Ausreißer existieren |

**Goldene Regel:** Immer normalisieren/standardisieren bevor du Distanzen berechnest – sonst dominieren Features mit großen Skalen das Ergebnis!